In [ ]:
#FPGA notebook
from tqdm.notebook import tqdm
from qick import *
#%pylab inline
#import h5py
#import json
#from scipy.signal import find_peaks
import zmq
import numpy as np
import threading
import traceback

In [ ]:
soc = QickSoc()
soccfg = soc
print(soccfg)

### Hardware Configuration

You should cable the board in loopback, connecting one DAC to one ADC. You can set appropriate generator and readout numbers in the cell below, which will be used in `res_ch` and `ro_chs` entries in the program dictionaries below.

- start with nothing

In [ ]:
class LoopbackProgram(AveragerProgram): #inherit from AveragerProgram
    def initialize(self):
        cfg=self.cfg #get config
        res_ch = cfg["res_ch"] #DAC channel select
        # configure DAC - set the nyquist zone
        self.declare_gen(ch=cfg["res_ch"], nqz=1)        

        # configure ADC - set the readout lengths and downconversion frequencies (ensuring it is an available DAC frequency)
        # 
        for ch in cfg["ro_chs"]:        
            self.declare_readout(
                ch=ch, 
                length=self.cfg["readout_length"],
                freq=self.cfg["pulse_freq"], 
                gen_ch=cfg["res_ch"]
            )

        # convert frequency to DAC frequency (ensuring it is an available ADC frequency)
        freq = self.freq2reg(cfg["pulse_freq"],gen_ch=res_ch, ro_ch=cfg["ro_chs"][0])
        phase = self.deg2reg(cfg["res_phase"], gen_ch=res_ch)
        gain = cfg["pulse_gain"]
        self.default_pulse_registers(ch=res_ch, freq=freq, phase=phase, gain=gain)

        # configure pulse style (ie. square, sinc, etc.)
        style=self.cfg["pulse_style"]

        if style in ["flat_top","arb"]:
            sigma = cfg["sigma"]
            self.add_gauss(ch=res_ch, name="measure", sigma=sigma, length=sigma*5)
        if style == "const":
            self.set_pulse_registers(ch=res_ch, style=style, length=cfg["length"])
        elif style == "flat_top":
            # The first half of the waveform ramps up the pulse, the second half ramps down the pulse
            self.set_pulse_registers(ch=res_ch, style=style, waveform="measure", length=cfg["length"])
        elif style == "arb":
            self.set_pulse_registers(ch=res_ch, style=style, waveform="measure")
        
        self.synci(200)  # give processor some time to configure pulses [clock ticks]
    
    def body(self):
        # fire the pulse
        # trigger all declared ADCs
        # pulse PMOD0_0 for a scope trigger
        # pause the tProc until readout is done
        # increment the time counter to give some time before the next measurement
        # (the syncdelay also lets the tProc get back ahead of the clock)
        
        # self.measure(pulse_ch=self.cfg["res_ch"], 
        #              adcs=self.ro_chs,
        #              pins=[0], 
        #              adc_trig_offset=self.cfg["adc_trig_offset"],
        #              wait=True,
        #              syncdelay=self.us2cycles(self.cfg["relax_delay"]))
        
        # equivalent to the following:
        self.trigger(adcs=self.ro_chs,
                     pins=[0], 
                     adc_trig_offset=self.cfg["adc_trig_offset"])
        self.pulse(ch=self.cfg["res_ch"])
        self.wait_all()
        self.sync_all(self.us2cycles(self.cfg["relax_delay"]))

In [ ]:
class FPGAController:

    def __init__(self):
        self.config = None
        self.frequency = None

    def configure(self, config):
        self.config = config
        print("FPGA configured:")
        print(config)

    def set_frequency(self, frequency):
        if self.config is None:
            raise RuntimeError("FPGA has not been configured")
        self.frequency = frequency
        self.config["pulse_freq"] = frequency
        print(f"Frequency set to {frequency} Hz")

    def measure(self):
        if self.config is None:
            raise RuntimeError("FPGA has not been configured")

        if self.frequency is None:
            raise RuntimeError("Frequency has not been set")

        prog = LoopbackProgram(soccfg, self.config)
        adc = prog.acquire_decimated(soc, progress=False)
        #adc = prog.acquire(soc, progress=False) how to use? supposedly quicker

        I = adc[:, 0]
        Q = adc[:, 1]

        return I.astype(np.float32) + 1j * Q.astype(np.float32)
    
    def time_axis(self):
        if self.config is None:
            raise RuntimeError("FPGA has not been configured")

        return np.array(
            soccfg.cycles2us(
                np.arange(self.config["readout_length"]),
                gen_ch=self.config["res_ch"]
            ),
            dtype=np.float32
        )

    def status(self):
        return {
            "connected": True,
            "frequency": self.frequency,
            "configured": self.config is not None
        }


class FPGAServer:

    def __init__(self, fpga, port=5555):
        self.fpga = fpga
        self.running = threading.Event()
        self.running.set()

        self.context = zmq.Context()
        self.socket = self.context.socket(zmq.REP)

        try:
            self.socket.bind(f"tcp://0.0.0.0:{port}")
        except zmq.ZMQError:
            self.socket.close(0)
            self.context.term()
            raise

        self.poller = zmq.Poller()
        self.poller.register(self.socket, zmq.POLLIN)

        print(f"FPGA server listening on port {port}")

    def run(self):
        try:
            while self.running.is_set():
                events = dict(self.poller.poll(100))

                if self.socket not in events:
                    continue

                try:
                    request = self.socket.recv_json()
                    command = request.get("command")
                    parameters = request.get("parameters", {})

                    print(f"Received command: {command}")

                    if command == "configure":
                        self.fpga.configure(parameters["config"])
                        self.socket.send_json({"status": "ok"})

                    elif command == "set_frequency":
                        self.fpga.set_frequency(parameters["frequency"])
                        self.socket.send_json({"status": "ok"})

                    elif command == "measure":
                        iq = self.fpga.measure()
                        self.socket.send(iq.astype(np.complex64).tobytes())
                        
                    elif command == "status":
                        self.socket.send_json({
                            "status": "ok",
                            "data": self.fpga.status()
                        })

                    elif command == "stop":
                        print("Shutdown command received")
                        self.socket.send_json({"status": "ok"})
                        self.running.clear()
                        
                    elif command == "time_axis":
                        time_axis = self.fpga.time_axis()
                        self.socket.send(time_axis.astype(np.float32).tobytes())
                        
                    else:
                        self.socket.send_json({
                            "status": "error",
                            "error": f"Unknown command: {command}"
                        })

                except Exception as e:
                    print("\nERROR HANDLING FPGA REQUEST:")
                    traceback.print_exc()

                    try:
                        self.socket.send_json({
                            "status": "error",
                            "error": str(e)
                        })
                    except zmq.ZMQError:
                        print("Could not send error response")

        except KeyboardInterrupt:
            print("\nKeyboard interrupt received")

        finally:
            self.stop()

    def stop(self):
        if self.running.is_set():
            print("Stopping FPGA server...")
            self.running.clear()

        try:
            self.poller.unregister(self.socket)
        except Exception:
            pass

        try:
            self.socket.close(0)
        except Exception:
            pass

        try:
            self.context.term()
        except Exception:
            pass

        print("FPGA server stopped")


if __name__ == "__main__":
    fpga = FPGAController()
    server = FPGAServer(fpga, port=5555)

    try:
        server.run()
    finally:
        server.stop()

